<a href="https://colab.research.google.com/github/fehmidataj27-dotcom/2nd-Task-/blob/main/AG_news_bert_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install transformers datasets scikit-learn torch

In [ ]:
import torch
from datasets import load_dataset
from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
dataset = load_dataset("ag_news")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(8000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(2000))

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=True,
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","label"]
)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "f1": f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    save_strategy="no"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.331370,0.321668,0.898500,0.898196


TrainOutput(global_step=250, training_loss=0.47333914947509764, metrics={'train_runtime': 85.5354, 'train_samples_per_second': 93.529, 'train_steps_per_second': 2.923, 'total_flos': 264944246784000.0, 'train_loss': 0.47333914947509764, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.3216676712036133,
 'eval_accuracy': 0.8985,
 'eval_f1': 0.8981962826849461,
 'eval_runtime': 6.8547,
 'eval_samples_per_second': 291.769,
 'eval_steps_per_second': 9.191,
 'epoch': 1.0}

In [ ]:
labels = ["World","Sports","Business","Sci/Tech"]

def predict(text):

    inputs = tokenizer(text, return_tensors="pt")

    outputs = model(**inputs)

    pred = torch.argmax(outputs.logits).item()

    return labels[pred]

In [ ]:
# Redefining the predict function within this cell to apply the fix locally
def predict(text):
    inputs = tokenizer(text, return_tensors="pt")
    # Move all input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits).item()
    return labels[pred]

predict("Apple launches new AI processor")

'Sci/Tech'

In [ ]:
model.save_pretrained("news_classifier")
tokenizer.save_pretrained("news_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('news_classifier/tokenizer_config.json', 'news_classifier/tokenizer.json')

In [ ]:
import streamlit as st
import torch
from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained("news_classifier")
tokenizer = DistilBertTokenizerFast.from_pretrained("news_classifier")

labels = ["World","Sports","Business","Sci/Tech"]

st.title("News Topic Classifier")

text = st.text_input("Enter News Headline")

if st.button("Predict"):

    inputs = tokenizer(text, return_tensors="pt")

    outputs = model(**inputs)

    pred = torch.argmax(outputs.logits).item()

    st.success(labels[pred])

# Save the Streamlit app code to a file
with open('app.py', 'w') as f:
    f.write("import streamlit as st\n")
    f.write("import torch\n")
    f.write("from transformers import DistilBertTokenizerFast\n")
    f.write("from transformers import DistilBertForSequenceClassification\n\n")
    f.write("model = DistilBertForSequenceClassification.from_pretrained(\"news_classifier\")\n")
    f.write("tokenizer = DistilBertTokenizerFast.from_pretrained(\"news_classifier\")\n\n")
    f.write("labels = [\"World\",\"Sports\",\"Business\",\"Sci/Tech\"]\n\n")
    f.write("st.title(\"News Topic Classifier\")\n\n")
    f.write("text = st.text_input(\"Enter News Headline\")\n\n")
    f.write("if st.button(\"Predict\"):\n\n")
    f.write("    inputs = tokenizer(text, return_tensors=\"pt\")\n\n")
    f.write("    outputs = model(**inputs)\n\n")
    f.write("    pred = torch.argmax(outputs.logits).item()\n\n")
    f.write("    st.success(labels[pred])\n")

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 

In [ ]:
!streamlit run app.py

In [ ]:
!pip install streamlit
!npm install -g localtunnel

In [ ]:
%%writefile app.py

import streamlit as st
import torch
from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained("news_classifier")
tokenizer = DistilBertTokenizerFast.from_pretrained("news_classifier")

labels = ["World","Sports","Business","Sci/Tech"]

st.title("News Topic Classifier")

text = st.text_input("Enter News Headline")

if st.button("Predict"):

    inputs = tokenizer(text, return_tensors="pt")

    outputs = model(**inputs)

    pred = torch.argmax(outputs.logits).item()

    st.success(labels[pred])

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!npx localtunnel --port 8501

In [ ]:
import gradio as gr

def classify(text):
    return predict(text)

gr.Interface(
    fn=classify,
    inputs="text",
    outputs="text",
    title="News Topic Classifier"
).launch()